# J-Reflect Recovery: Colab A100 runner

1. **Runtime → Change runtime type → A100 GPU**, and turn on **High-RAM**.
2. **Runtime → Run all.** Approve the Google Drive prompt.

All progress is copied to `MyDrive/jrecovery/runs/pilot` every 5 minutes and after every step.
**If Colab disconnects, just Run all again**: completed records are skipped.

Execution stops on the first failure. This includes the development **freeze gate**: if it stops there, send the dev report to Claude before changing anything.

In [ ]:
import os, subprocess
from google.colab import drive
from IPython.display import Markdown, display

drive.mount('/content/drive')

COMMIT = "c1937e1"  # exact code version; change only together with a new run directory
REPO   = "/content/J-space-Ignition/J-Reflect-Recovery"
DRIVE  = "/content/drive/MyDrive/jrecovery"
RUN    = "/content/runs/pilot"
SYNC   = f"rsync -a --exclude '*.tmp' --exclude .lock /content/runs/ {DRIVE}/runs/"

def sh(cmd):
    """Run a shell command, stream its output, and raise if it fails."""
    p = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")

def jr(*args):
    """Run a jrecovery command, then always copy the run to Drive."""
    try:
        sh(f"cd {REPO} && .venv/bin/jrecovery " + " ".join(args))
    finally:
        sh(SYNC)

def show(split):
    display(Markdown(open(f"{RUN}/reports/{split}/report.md").read()))

In [ ]:
# Setup: same code, same Python, same packages every session (the run refuses to continue otherwise).
sh(f'''
set -e
mkdir -p {DRIVE}/runs /content/runs
command -v rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync >/dev/null)
cd /content
[ -d J-space-Ignition ] || git clone -q https://github.com/ljbcoder/J-space-Ignition.git
cd J-space-Ignition && git fetch -q && git checkout -q {COMMIT}
cd J-Reflect-Recovery
pip install -q uv
PY=$(cat {DRIVE}/python-version 2>/dev/null || echo 3.12)
uv sync --locked --extra dev --extra model --managed-python --python "$PY"
.venv/bin/python -c "import platform; print(platform.python_version())" > {DRIVE}/python-version
echo "Python $(cat {DRIVE}/python-version)"
rsync -a {DRIVE}/runs/ /content/runs/
.venv/bin/jrecovery doctor
.venv/bin/pytest -q
''')

In [ ]:
# Background copy to Drive every 5 minutes.
subprocess.Popen(["bash", "-c", f"while true; do {SYNC}; sleep 300; done"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
print("Background Drive sync started")

## 1. Development split (200 problems)

In [ ]:
if not os.path.exists(f"{RUN}/run.json"):
    jr("prepare", "--config", "configs/pilot.toml", "--run", RUN)
else:
    print("Run already prepared; resuming")

# Smoke test first (downloads ~19 GB of weights on first use), then the rest.
jr("collect", "--run", RUN, "--split", "dev", "--limit", "10")
jr("collect", "--run", RUN, "--split", "dev")
jr("report",  "--run", RUN, "--split", "dev")
show("dev")

In [ ]:
jr("correct", "--run", RUN, "--split", "dev")
jr("ablate",  "--run", RUN, "--split", "dev")
jr("report",  "--run", RUN, "--split", "dev")
show("dev")

## 2. Freeze gate

Requires at least 10 eligible wrong first attempts, and a J++ hit rate above both the logit-lens and shuffled baselines.
If this cell fails, **stop**. Do not edit the config inside this run: a harder-difficulty retry needs a new run directory.

In [ ]:
if not os.path.exists(f"{RUN}/frozen.json"):
    jr("freeze", "--run", RUN)
else:
    print("Already frozen")

## 3. Held-out test split (1,000 problems)

In [ ]:
jr("collect", "--run", RUN, "--split", "test")
jr("correct", "--run", RUN, "--split", "test")
jr("ablate",  "--run", RUN, "--split", "test")
jr("report",  "--run", RUN, "--split", "test")
show("test")

In [ ]:
sh(SYNC)
print(f"Done. Reports are in {DRIVE}/runs/pilot/reports/")